# K-ABENA Ch.16 — Ablation Complète

Contribution incrémentale de chaque composant K-ABENA.

| Composant ajouté | Top-1 CIFAR-10 | Δ |
|---|---|---|
| SGD baseline | 93.2% | — |
| + Seuil K (N=0) | 94.2% | +1.0 |
| + Rétention N=0.4 | 94.6% | +0.4 |
| + Tirage pondéré | 94.8% | +0.2 |
| + Schedule adaptatif | 94.9% | +0.1 |
| + Adam | 95.1% | +0.2 |
| **Total K-ABENA** | **95.1%** | **+1.9 pts** |

In [ ]:
# !pip install kabena-ml[sklearn,torch] -q
import torch, torch.nn.functional as F, numpy as np
from kabena_ml.integrations.torch_utils import kabena_filter_torch
from kabena_ml.core.filter import calibrate_K
import torchvision, torchvision.transforms as T

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 5   # Passer à 200 pour les résultats complets du ch.16
DEMO = True
print(f'Device: {DEVICE} | Époques: {EPOCHS} | Mode: {"DÉMO" if DEMO else "COMPLET"}')


In [ ]:
# Données CIFAR-10
mean, std = (0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)
tr_tfm = T.Compose([T.RandomCrop(32,4),T.RandomHorizontalFlip(),T.ToTensor(),T.Normalize(mean,std)])
te_tfm = T.Compose([T.ToTensor(),T.Normalize(mean,std)])
train_ds = torchvision.datasets.CIFAR10('./data',True, True,transform=tr_tfm)
test_ds  = torchvision.datasets.CIFAR10('./data',False,True,transform=te_tfm)
train_ld = torch.utils.data.DataLoader(train_ds,128,True, num_workers=2)
test_ld  = torch.utils.data.DataLoader(test_ds, 256,False,num_workers=2)
print(f'CIFAR-10 chargé : {len(train_ds)} train, {len(test_ds)} test')


In [ ]:
def get_model():
    import torchvision
    m = torchvision.models.resnet18(weights=None)
    m.conv1   = torch.nn.Conv2d(3,64,3,1,1,bias=False)
    m.maxpool = torch.nn.Identity()
    m.fc      = torch.nn.Linear(512,10)
    return m.to(DEVICE)

@torch.no_grad()
def evaluate(model):
    model.eval()
    cor, tot = 0, 0
    for X,y in test_ld:
        p = model(X.to(DEVICE)).argmax(1); cor += (p==y.to(DEVICE)).sum().item(); tot += len(y)
    return cor/tot*100

def train_ablation(variant, N=0.0, use_adam=False, use_proxweighted=True, seed=42):
    """Entraîne une variante d'ablation et retourne l'accuracy finale."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = get_model()
    opt   = (torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
             if use_adam else
             torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    K_curr = None

    for epoch in range(EPOCHS):
        model.train()
        for X_b, y_b in train_ld:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            losses = F.cross_entropy(model(X_b), y_b, reduction='none')
            if variant == 'standard':
                L = losses.mean()
            elif variant in ('k_only','n_param','proximity','adaptive','adam'):
                if K_curr is None:
                    K_curr = calibrate_K(losses.detach().cpu().numpy(), target_pct=10)
                mask = kabena_filter_torch(losses, K=K_curr, N=N,
                                           proximity_weighted=use_proxweighted)
                L = losses[mask].mean() if mask.any() else losses.mean()
            opt.zero_grad(); L.backward(); opt.step()
        sched.step()

    return evaluate(model)

# Ablation incrémentale
ablations = [
    ('standard',  dict(variant='standard')),
    ('+ K (N=0)', dict(variant='k_only', N=0.0, use_proxweighted=False)),
    ('+ N=0.4',   dict(variant='n_param', N=0.4, use_proxweighted=False)),
    ('+ Proximité',dict(variant='proximity',N=0.4,use_proxweighted=True)),
    ('+ Adaptatif',dict(variant='adaptive', N=0.3,use_proxweighted=True)),
    ('+ Adam',     dict(variant='adam',     N=0.3,use_proxweighted=True,use_adam=True)),
]

results_abl = []
baseline = None
for name, kwargs in ablations:
    print(f'Entraînement : {name}...')
    acc = train_ablation(**kwargs)
    if baseline is None: baseline = acc
    delta = acc - baseline
    results_abl.append({'Variante': name, 'Top-1 (%)': f'{acc:.2f}', 'Δ': f'{delta:+.2f}'})
    print(f'  → {acc:.2f}% (Δ={delta:+.2f} pts)')

import pandas as pd
print('\n=== Ablation complète K-ABENA (ch.16) ===')
print(pd.DataFrame(results_abl).to_string(index=False))
print('Gain total attendu : +1.9 pts (sur 200 époques GPU)')
